In [11]:
import time
import requests
from bs4 import BeautifulSoup

def extrair_links_top_anime(paginas_adicionais=2):
    url_base = "https://myanimelist.net/topanime.php"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    todos_os_links = []

    urls_para_raspar = [url_base]
    for i in range(1, paginas_adicionais + 1):
        limite = i * 50
        urls_para_raspar.append(f"{url_base}?limit={limite}")

    for url in urls_para_raspar:
        try:
            resposta = requests.get(url, headers=headers, timeout=10)

            if resposta.status_code == 200:
                soup = BeautifulSoup(resposta.text, "html.parser")
                linhas_ranking = soup.find_all("tr", class_="ranking-list")

                for linha in linhas_ranking:
                    h3_tag = linha.find("h3", class_="anime_ranking_h3")
                    
                    if h3_tag:
                        link_anime = h3_tag.find("a")
                        if link_anime:
                            href = link_anime.get("href")
                            texto = link_anime.text.strip()

                            if href and (href, texto) not in todos_os_links:
                                todos_os_links.append((href, texto))

        except Exception:
            # Ignora erros silenciosamente para não sujar a tela
            pass

        # Pausa para não sobrecarregar o servidor
        time.sleep(2)

    return todos_os_links


# --- Executando o Scraping ---
animes_encontrados = extrair_links_top_anime(paginas_adicionais=20)

# Imprime APENAS os 150 animes e seus links
for idx, (link, titulo) in enumerate(animes_encontrados, start=1):
    print(f"{idx}. {titulo} -> {link}")

1. Sousou no Frieren -> https://myanimelist.net/anime/52991/Sousou_no_Frieren
2. Steel Ball Run: JoJo no Kimyou na Bouken -> https://myanimelist.net/anime/61469/Steel_Ball_Run__JoJo_no_Kimyou_na_Bouken
3. Fullmetal Alchemist: Brotherhood -> https://myanimelist.net/anime/5114/Fullmetal_Alchemist__Brotherhood
4. Chainsaw Man Movie: Reze-hen -> https://myanimelist.net/anime/57555/Chainsaw_Man_Movie__Reze-hen
5. Steins;Gate -> https://myanimelist.net/anime/9253/Steins_Gate
6. Gintama: The Final -> https://myanimelist.net/anime/39486/Gintama__The_Final
7. Gintama° -> https://myanimelist.net/anime/28977/Gintama°
8. Shingeki no Kyojin Season 3 Part 2 -> https://myanimelist.net/anime/38524/Shingeki_no_Kyojin_Season_3_Part_2
9. Hunter x Hunter (2011) -> https://myanimelist.net/anime/11061/Hunter_x_Hunter_2011
10. Ginga Eiyuu Densetsu -> https://myanimelist.net/anime/820/Ginga_Eiyuu_Densetsu
11. Gintama' -> https://myanimelist.net/anime/9969/Gintama
12. Gintama': Enchousen -> https://myanimelist

In [12]:
for idx, (link, titulo) in enumerate(animes_encontrados, start=1):
    print(f"{idx}. {titulo} -> {link}")

1. Sousou no Frieren -> https://myanimelist.net/anime/52991/Sousou_no_Frieren
2. Steel Ball Run: JoJo no Kimyou na Bouken -> https://myanimelist.net/anime/61469/Steel_Ball_Run__JoJo_no_Kimyou_na_Bouken
3. Fullmetal Alchemist: Brotherhood -> https://myanimelist.net/anime/5114/Fullmetal_Alchemist__Brotherhood
4. Chainsaw Man Movie: Reze-hen -> https://myanimelist.net/anime/57555/Chainsaw_Man_Movie__Reze-hen
5. Steins;Gate -> https://myanimelist.net/anime/9253/Steins_Gate
6. Gintama: The Final -> https://myanimelist.net/anime/39486/Gintama__The_Final
7. Gintama° -> https://myanimelist.net/anime/28977/Gintama°
8. Shingeki no Kyojin Season 3 Part 2 -> https://myanimelist.net/anime/38524/Shingeki_no_Kyojin_Season_3_Part_2
9. Hunter x Hunter (2011) -> https://myanimelist.net/anime/11061/Hunter_x_Hunter_2011
10. Ginga Eiyuu Densetsu -> https://myanimelist.net/anime/820/Ginga_Eiyuu_Densetsu
11. Gintama' -> https://myanimelist.net/anime/9969/Gintama
12. Gintama': Enchousen -> https://myanimelist

In [13]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd # Para salvar os dados no final

def extrair_detalhes_anime(links_animes):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    dados_completos = []

    print(f"Iniciando a extração de {len(links_animes)} animes...")

    for indice, (url, titulo) in enumerate(links_animes, start=1):
        print(f"[{indice}/{len(links_animes)}] Extraindo: {titulo}")
        
        # Dicionário padrão preenchido com 'NA'
        anime_data = {
            "Title": titulo,
            "URL": url,
            "Image_URL": "NA",
            "Synopsis": "NA",
            "Type": "NA",
            "Episodes": "NA",
            "Status": "NA",
            "Aired": "NA",
            "Premiered": "NA",
            "Broadcast": "NA",
            "Producers": "NA",
            "Licensors": "NA",
            "Studios": "NA",
            "Source": "NA",
            "Genres": "NA",
            "Demographic": "NA",
            "Duration": "NA",
            "Rating": "NA",
            "Score": "NA",
            "Ranked": "NA",
            "Popularity": "NA",
            "Members": "NA",
            "Favorites": "NA"
        }

        try:
            resposta = requests.get(url, headers=headers, timeout=10)

            if resposta.status_code == 200:
                soup = BeautifulSoup(resposta.text, "html.parser")

                # 1. Extraindo a Imagem Principal
                img_tag = soup.find("img", itemprop="image")
                if img_tag:
                    # Tenta pegar o data-src primeiro (usado para lazy loading), senão pega o src
                    anime_data["Image_URL"] = img_tag.get("data-src") or img_tag.get("src", "NA")

                # 2. Extraindo a Sinopse
                synopsis_tag = soup.find(itemprop="description")
                if synopsis_tag:
                    # Pega o texto, removendo espaços e quebras de linha excessivas
                    anime_data["Synopsis"] = " ".join(synopsis_tag.text.split())

                # 3. Extraindo Information e Statistics (Barra Lateral)
                # O MAL usa a classe 'dark_text' para os títulos (ex: "Type:", "Episodes:")
                dark_texts = soup.find_all("span", class_="dark_text")
                
                for span in dark_texts:
                    chave = span.text.strip().replace(":", "")
                    
                    if chave in anime_data:
                        # O valor é o resto do texto do elemento 'pai', excluindo o título (span)
                        pai = span.parent
                        valor = pai.text.replace(span.text, "").strip()
                        # Limpa quebras de linha para ficar reto no banco de dados
                        valor = " ".join(valor.split())
                        
                        # Atribui se não for vazio
                        if valor:
                            anime_data[chave] = valor

            else:
                print(f"  -> Erro ao acessar: HTTP {resposta.status_code}")

        except Exception as e:
            print(f"  -> Falha na requisição: {e}")

        # Adiciona o dicionário populado (com os dados ou os NAs) à lista final
        dados_completos.append(anime_data)

        # Pausa dramática para o MyAnimeList não nos bloquear
        time.sleep(2)

    return dados_completos


# ==========================================
# TESTE DO SCRIPT E CRIAÇÃO DO BANCO DE DADOS
# ==========================================

# Simulação da lista que você obteve no código anterior
# Substitua esta variável 'links_coletados' pela sua lista original gerada antes!
links_coletados = animes_encontrados

# Executa o scraping detalhado
dados_extraidos = extrair_detalhes_anime(links_coletados)

# Exibe o resultado do primeiro para conferência
print("\n--- Exemplo do primeiro extraído ---")
for chave, valor in dados_extraidos[0].items():
    print(f"{chave}: {valor}")

# ==========================================
# EXPORTANDO PARA O BANCO DE DADOS
# ==========================================
# Converte a lista de dicionários em uma tabela do Pandas e salva em CSV
df_animes = pd.DataFrame(dados_extraidos)
df_animes.to_csv("banco_de_dados_animes.csv", index=False, encoding="utf-8-sig")

print("\nSucesso! O banco de dados foi salvo no arquivo 'banco_de_dados_animes.csv'.")

Iniciando a extração de 1050 animes...
[1/1050] Extraindo: Sousou no Frieren
[2/1050] Extraindo: Steel Ball Run: JoJo no Kimyou na Bouken
[3/1050] Extraindo: Fullmetal Alchemist: Brotherhood
[4/1050] Extraindo: Chainsaw Man Movie: Reze-hen
[5/1050] Extraindo: Steins;Gate
[6/1050] Extraindo: Gintama: The Final
[7/1050] Extraindo: Gintama°
[8/1050] Extraindo: Shingeki no Kyojin Season 3 Part 2
[9/1050] Extraindo: Hunter x Hunter (2011)
[10/1050] Extraindo: Ginga Eiyuu Densetsu
[11/1050] Extraindo: Gintama'
[12/1050] Extraindo: Gintama': Enchousen
[13/1050] Extraindo: One Piece Fan Letter
[14/1050] Extraindo: Bleach: Sennen Kessen-hen
[15/1050] Extraindo: Gintama.
[16/1050] Extraindo: Kaguya-sama wa Kokurasetai: Ultra Romantic
[17/1050] Extraindo: Clannad: After Story
[18/1050] Extraindo: Fruits Basket: The Final
[19/1050] Extraindo: Gintama
[20/1050] Extraindo: Koe no Katachi
[21/1050] Extraindo: Code Geass: Hangyaku no Lelouch R2
[22/1050] Extraindo: Kusuriya no Hitorigoto 2nd Season
[2

In [14]:
import pandas as pd
import numpy as np
import re

# ==========================================
# EXPORTANDO PARA O BANCO DE DADOS
# ==========================================
# Converte a lista de dicionários em uma tabela do Pandas
df_animes = pd.DataFrame(dados_extraidos)

# --- INÍCIO DA LIMPEZA DOS DADOS ---

# 1. Limpeza da coluna 'Score'
# Extrai apenas os números no formato X.XX (ignorando o "1" do rodapé e o texto de usuários)
if 'Score' in df_animes.columns:
    df_animes['Score'] = df_animes['Score'].str.extract(r'^(\d+\.\d{2})').astype(float)

# 2. Limpeza da coluna 'Ranked'
# Extrai apenas os números logo após o símbolo de hashtag "#", ignorando o parágrafo de texto
if 'Ranked' in df_animes.columns:
    df_animes['Ranked'] = df_animes['Ranked'].str.extract(r'#(\d+)').astype(float)

# 3. Remoção de Duplicação Literal (ex: "AdventureAdventure" -> "Adventure")
def limpar_texto_duplicado(texto):
    if pd.isna(texto) or not isinstance(texto, str):
        return texto
    
    itens_limpos = []
    # Separa por vírgula (caso existam múltiplos gêneros)
    for item in texto.split(','):
        item = item.strip()
        meio = len(item) // 2
        
        # Verifica se a primeira metade da palavra é exatamente igual à segunda metade
        if len(item) > 1 and item[:meio] == item[meio:]:
            itens_limpos.append(item[:meio]) # Mantém só uma metade
        else:
            itens_limpos.append(item)
            
    return ', '.join(itens_limpos)

if 'Genres' in df_animes.columns:
    df_animes['Genres'] = df_animes['Genres'].apply(limpar_texto_duplicado)
if 'Demographic' in df_animes.columns:
    df_animes['Demographic'] = df_animes['Demographic'].apply(limpar_texto_duplicado)

# 4. Tratamento de Nulos (Nulls)
# Substitui o botão de adicionar informação do site por nulos (NaN) reais do sistema
df_animes.replace(r'(?i).*None found, add some.*', np.nan, regex=True, inplace=True)
df_animes.replace(["Unknown", "NA"], np.nan, inplace=True)

# --- FIM DA LIMPEZA DOS DADOS ---

# Salva em CSV o banco de dados limpo
df_animes.to_csv("banco_de_dados_animes_limpo.csv", index=False, encoding="utf-8-sig")

print("\nSucesso! O banco de dados foi limpo e salvo no arquivo 'banco_de_dados_animes_limpo.csv'.")


Sucesso! O banco de dados foi limpo e salvo no arquivo 'banco_de_dados_animes_limpo.csv'.


In [1]:
import pandas as pd
import cv2
import numpy as np
import requests
import time

def extrair_caracteristicas_cor_e_intensidade(url_imagem):
    """
    Baixa uma imagem da URL e extrai um vetor de características baseado 
    no histograma RGB e na intensidade da imagem.
    """
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resposta = requests.get(url_imagem, headers=headers, stream=True, timeout=10)
        if resposta.status_code != 200:
            return None
            
        imagem_array = np.asarray(bytearray(resposta.content), dtype=np.uint8)
        img = cv2.imdecode(imagem_array, cv2.IMREAD_COLOR) 
        
        if img is None:
            return None

        img = cv2.resize(img, (150, 150))
        b_canal, g_canal, r_canal = cv2.split(img)
        
        bins = 8
        hist_r = cv2.calcHist([r_canal], [0], None, [bins], [0, 256])
        hist_g = cv2.calcHist([g_canal], [0], None, [bins], [0, 256])
        hist_b = cv2.calcHist([b_canal], [0], None, [bins], [0, 256])
        
        hist_r = cv2.normalize(hist_r, hist_r).flatten()
        hist_g = cv2.normalize(hist_g, hist_g).flatten()
        hist_b = cv2.normalize(hist_b, hist_b).flatten()
        
        img_cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        media_intensidade = np.mean(img_cinza) / 255.0  
        desvio_intensidade = np.std(img_cinza) / 255.0  
        
        vetor = np.concatenate([
            hist_r, 
            hist_g, 
            hist_b, 
            [media_intensidade, desvio_intensidade]
        ])
        return vetor

    except Exception:
        return None

# ==========================================
# PROCESSANDO O DATASET INTEIRO
# ==========================================

# 1. Carrega o dataset limpo
df_animes = pd.read_csv("banco_de_dados_animes_limpo.csv")

# 2. Cria a variável que vai armazenar os vetores de cada imagem
vetor_imagem = []

print(f"Iniciando a extração de vetores para {len(df_animes)} animes...")

# 3. Itera sobre cada linha do banco de dados
for indice, linha in df_animes.iterrows():
    url = linha['Image_URL']
    titulo = linha['Title']
    
    # Se não tiver URL, adiciona None para manter o tamanho da lista alinhado com o dataset
    if pd.isna(url) or url == "NA":
        print(f"[{indice+1}/{len(df_animes)}] Pulado (sem imagem): {titulo}")
        vetor_imagem.append(None)
        continue
        
    # Extrai o vetor
    vetor = extrair_caracteristicas_cor_e_intensidade(url)
    vetor_imagem.append(vetor)
    
    if vetor is not None:
        print(f"[{indice+1}/{len(df_animes)}] Vetor extraído com sucesso: {titulo}")
    else:
        print(f"[{indice+1}/{len(df_animes)}] Falha ao extrair: {titulo}")
        
    # Pausa de 0.5 segundos para o MyAnimeList não bloquear nosso IP
    time.sleep(0.5)

# 4. Adiciona a variável de volta ao DataFrame original como uma nova coluna
df_animes['vetor_imagem'] = vetor_imagem

# 5. Salva o DataFrame como Pickle (.pkl) para preservar os Arrays do NumPy intactos!
df_animes.to_pickle("banco_de_dados_animes_vetorizado.pkl")

print("\nProcesso 100% concluído!")
print("Os dados agora possuem a coluna 'vetor_imagem' e foram salvos no arquivo 'banco_de_dados_animes_vetorizado.pkl'.")

Iniciando a extração de vetores para 1050 animes...
[1/1050] Vetor extraído com sucesso: Sousou no Frieren
[2/1050] Vetor extraído com sucesso: Steel Ball Run: JoJo no Kimyou na Bouken
[3/1050] Vetor extraído com sucesso: Fullmetal Alchemist: Brotherhood
[4/1050] Vetor extraído com sucesso: Chainsaw Man Movie: Reze-hen
[5/1050] Vetor extraído com sucesso: Steins;Gate
[6/1050] Vetor extraído com sucesso: Gintama: The Final
[7/1050] Vetor extraído com sucesso: Gintama°
[8/1050] Vetor extraído com sucesso: Shingeki no Kyojin Season 3 Part 2
[9/1050] Vetor extraído com sucesso: Hunter x Hunter (2011)
[10/1050] Vetor extraído com sucesso: Ginga Eiyuu Densetsu
[11/1050] Vetor extraído com sucesso: Gintama'
[12/1050] Vetor extraído com sucesso: Gintama': Enchousen
[13/1050] Vetor extraído com sucesso: One Piece Fan Letter
[14/1050] Vetor extraído com sucesso: Bleach: Sennen Kessen-hen
[15/1050] Vetor extraído com sucesso: Gintama.
[16/1050] Vetor extraído com sucesso: Kaguya-sama wa Kokuraset